# Colab Training via VS Code Extension

Make sure you have connected this notebook to a **Google Colab Kernel** using the top-right kernel selector.


In [13]:
%cd /

/


In [14]:
# 1. Clone Codebase from Git
import os
import shutil

# Remove the old clone so we get the fresh code with the qm9.py fix
if os.path.exists("/content/project"):
    shutil.rmtree("/content/project")

REPO_URL = "https://github.com/KoyaTofu42/cfg-molecular-diffusion.git"

%cd /
!git clone $REPO_URL /content/project
%cd /content/project

/
Cloning into '/content/project'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 101 (delta 19), reused 101 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 5.28 MiB | 12.56 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/project


In [2]:
# 2. Install dependencies
!pip install torch torchvision torchaudio -q
!pip install matplotlib numpy scipy rdkit-pypi wandb -q

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [3]:
# 3. Authenticate with Weights & Biases (WandB)
import wandb
import os
import getpass

# Using getpass prompts you in VSCode securely without saving the key in the notebook
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your WANDB_API_KEY: ")

wandb.login()

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
if not WANDB_ENTITY:
    WANDB_ENTITY = input(
        "Enter your WandB entity/username (account name only): "
    ).strip()
WANDB_ENTITY = WANDB_ENTITY.split("/", 1)[0].strip()
if not WANDB_ENTITY:
    raise ValueError("WANDB_ENTITY is required")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rk406 (rk406-01) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# 4. (Optional) Restore Checkpoint from WandB
# If your Colab runtime disconnected and you want to resume training:
# Set WANDB_RUN_PATH to a real run path like 'username/e3_diffusion/run-12345'.

WANDB_RUN_PATH = ""
RESUME_ARGS = ""  # Leave blank if starting from scratch

if WANDB_RUN_PATH:
    api = wandb.Api()
    run = api.run(WANDB_RUN_PATH)
    os.makedirs(
        "/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property",
        exist_ok=True,
    )
    for file in run.files():
        if file.name.endswith(".npy") or file.name.endswith(".pickle"):
            file.download(
                root="/content/project/e3_diffusion_for_molecules/outputs/cfg_multi_property",
                replace=True,
            )
    print("Restored checkpoint from WandB")
    run_id = WANDB_RUN_PATH.split("/")[-1]
    RESUME_ARGS = f"--resume outputs/cfg_multi_property --wandb_run_id {run_id}"

In [5]:
# 5. Download Dataset from WandB
import wandb
import os
import shutil

dataset_dir = "/content/project/e3_diffusion_for_molecules/qm9/temp/qm9"
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading QM9 dataset from WandB Artifacts...")
api = wandb.Api()
artifact = api.artifact(f"{WANDB_ENTITY}/e3_diffusion/qm9_raw:latest", type="dataset")
artifact.download(root=dataset_dir)
print("Dataset downloaded successfully!")

wandb: Downloading large artifact 'qm9_raw:latest', 82.62MB. 3 files...
wandb:   3 of 3 files downloaded.  
Done. 00:00:00.6 (132.2MB/s)


Dataset downloaded successfully!


In [15]:
# 6. Start Training!
!cd e3_diffusion_for_molecules && python main_qm9.py \
    --n_epochs 3000 \
    --exp_name cfg_multi_property \
    --conditioning alpha mu gap \
    --cfg_dropout_prob 0.15 \
    --cfg_dropout_mode joint \
    --n_stability_samples 500 \
    --diffusion_noise_schedule polynomial_2 \
    --diffusion_noise_precision 1e-5 \
    --diffusion_steps 1000 \
    --diffusion_loss_type l2 \
    --batch_size 32 \
    --nf 128 \
    --n_layers 6 \
    --lr 1e-4 \
    --normalize_factors "[1,4,10]" \
    --test_epochs 50 \
    --ema_decay 0.9999 \
    --save_model True \
    --wandb_usr "{WANDB_ENTITY}" \
    {RESUME_ARGS}

/content/project/e3_diffusion_for_molecules/qm9/data/prepare/process.py:126: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if line[0] is '#':
/content/project/e3_diffusion_for_molecules/qm9/data/prepare/process.py:128: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  if line_counter is 0:
/content/project/e3_diffusion_for_molecules/qm9/data/prepare/process.py:130: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  elif line_counter is 1:
/content/project/e3_diffusion_for_molecules/qm9/data/prepare/process.py:143: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  if len(split) is 4:
/content/project/e3_diffusion_for_molecules/qm9/data/args.py:183: SyntaxWarning: invalid escape sequence '\e'
  help='Concatenate the scalars from different \ell in the dot-product-matrix part of the edge network.')
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rk406 (rk406-01) to https://a